In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = "/mnt/data/masster_checkpoints/enveda_180_masster_s0/metrics.csv"
df = pd.read_csv(metrics_path)

# Lightning logs train and val metrics on separate rows, so drop NaNs per column.
train = df[["epoch", "train_loss_epoch/mean"]].dropna()
val = df[["epoch", "val_loss_epoch/mean"]].dropna()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train["epoch"], train["train_loss_epoch/mean"], label="train")
ax.plot(val["epoch"], val["val_loss_epoch/mean"], label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("enveda_180_masster_s0")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Similarity metrics. Validation logs val_<name>_epoch/mean for every auxiliary score; train
# logs the same only when the run set compute_train_auxiliary_metrics: true, so a missing
# train column is skipped.
sim_names = ["cos_sim_0.01", "precision", "recall", "wprecision", "wrecall"]
sim_names = [n for n in sim_names if f"val_{n}_epoch/mean" in df.columns]

fig, axes = plt.subplots(1, len(sim_names), figsize=(4 * len(sim_names), 3.5), squeeze=False)
for ax, name in zip(axes[0], sim_names):
    for split in ("train", "val"):
        col = f"{split}_{name}_epoch/mean"
        if col in df.columns:
            curve = df[["epoch", col]].dropna()
            ax.plot(curve["epoch"], curve[col], label=split)
    ax.set_xlabel("epoch")
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle(metrics_path.split("/")[-2])
fig.tight_layout()
plt.show()

# Last validated epoch of each metric; the best cosine epoch matches the saved checkpoint.
last = df[[f"val_{n}_epoch/mean" for n in sim_names]].dropna().iloc[-1]
print(last.to_string())
best_col = "val_cos_sim_0.01_epoch/mean"
if best_col in df.columns:
    best = df[["epoch", best_col]].dropna()
    best = best.loc[best[best_col].idxmax()]
    print(f"best {best_col}: {best[best_col]:.4f} at epoch {int(best['epoch'])}")
